# [17.1] Checkpoint Archaeology and Mechanism Emergence

> **Local-first extension.** This section teaches checkpoint archaeology as a bounded mechanism-emergence workflow. The exercises use deterministic toy checkpoint trajectories and a CPU-feasible live mod-13 model organism; the CUDA report reruns the same train/save/reload/control path on the RTX 5090.

## Core Question

When can we say a mechanism emerged during training?

A good answer is not just a rising metric. It needs a threshold, a stable run above the threshold, a phase-transition or timing report, saved checkpoints that can be reloaded, and a negative control that does not look like the same mechanism.

## Learning Objectives

By the end of this notebook you should be able to separate first crossing from stable crossing, count monotonicity violations, detect adjacent jumps, reject overstrong random controls, compare toy developmental timings without ranking the control, and read a CUDA checkpoint report without widening the finite-model-organism claim.


In [ ]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path

import torch as t

GT_TIER = "GT-0"
EXERCISE_ID = "17_1_checkpoint_archaeology_and_mechanism_emergence"
DIFFICULTY = 4
IMPORTANCE = 3
EXPECTED_RUNTIME = "seconds on toy contract; minutes on local real-model path"
REQUIRES_GPU = True

chapter = "chapter17_training_dynamics"
section = "part1_checkpoint_archaeology"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_checkpoint_archaeology.tests as tests
import part1_checkpoint_archaeology.utils as utils

from arena_ext.training_dynamics import (
    DevelopmentalComparisonReport,
    MechanismEmergenceReport,
    PhaseTransitionReport,
    RandomControlReport,
    toy_training_trajectories,
)


## Exercise 1 - Checkpoint Series Helpers

The smallest useful checkpoint-archaeology primitive is a validated series and two different crossing notions: first crossing and stable crossing.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_first_threshold_crossing_finds_first_crossing_and_validates_inputs` passed!
All tests in `test_stable_threshold_step_requires_consecutive_checkpoints` passed!
```

</details>

<details>
<summary>Help - how do first and stable crossings differ?</summary>

A first crossing can be a single lucky spike. A stable crossing starts a window of `min_consecutive` checkpoints that all stay above threshold.

</details>

<details>
<summary>Common bugs</summary>

- Returning the tensor index rather than the checkpoint step.
- Sorting non-monotone checkpoints instead of rejecting them.
- Using strict `>` and missing a value exactly equal to threshold.

</details>

<details>
<summary>Solution</summary>

Validate shape, monotone step order, and finite values. Use `values >= threshold`, then return the corresponding checkpoint step.

</details>


In [ ]:
def _validate_checkpoint_series(
    steps: t.Tensor,
    values: t.Tensor,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


def first_threshold_crossing(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    threshold: float,
) -> int | None:
    raise NotImplementedError()


def stable_threshold_step(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    threshold: float,
    min_consecutive: int = 2,
) -> int | None:
    raise NotImplementedError()


tests.test_first_threshold_crossing_finds_first_crossing_and_validates_inputs(
    first_threshold_crossing
)
tests.test_stable_threshold_step_requires_consecutive_checkpoints(stable_threshold_step)


## Exercise 2 - Emergence Reports

A report should make the claim auditable. Keep the threshold, first crossing, stable crossing, peak checkpoint, peak value, monotonicity violations, and final boolean.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_mechanism_emergence_report_tracks_peak_and_monotonicity` passed!
```

</details>

<details>
<summary>Help - what should the report make visible?</summary>

A reviewer should be able to tell whether the mechanism is stable, whether the peak came later, and whether the metric regressed between checkpoints.

</details>

<details>
<summary>Common bugs</summary>

- Reporting only first crossing and hiding stable crossing.
- Letting NaNs pass as evidence.
- Counting tolerated numerical noise as a real monotonicity failure.

</details>

<details>
<summary>Solution</summary>

Count adjacent decreases larger than tolerance, reuse the crossing helpers, compute the peak with `argmax`, and require stable crossing plus limited monotonicity violations.

</details>


In [ ]:
def monotonicity_violations(values: t.Tensor, *, tolerance: float = 0.0) -> int:
    raise NotImplementedError()


def mechanism_emergence_report(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    metric_name: str = "mechanism_metric",
    threshold: float = 0.6,
    min_consecutive: int = 2,
    max_monotonicity_violations: int = 1,
) -> MechanismEmergenceReport:
    raise NotImplementedError()


tests.test_mechanism_emergence_report_tracks_peak_and_monotonicity(
    mechanism_emergence_report
)


## Exercise 3 - Phase Transitions and Controls

The largest adjacent jump is a developmental warning, not proof. The random-control report is the falsification check: if the control trajectory also looks strong, the mechanism story should fail.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_phase_transition_report_detects_largest_adjacent_jump` passed!
All tests in `test_random_control_report_rejects_overstrong_control` passed!
```

</details>

<details>
<summary>Help - why adjacent jumps?</summary>

A phase-transition warning asks where the metric changed fastest between neighboring checkpoints. Comparing every checkpoint to step zero answers a different question.

</details>

<details>
<summary>Common bugs</summary>

- Returning the pre-jump checkpoint instead of the post-jump checkpoint.
- Checking only the final random-control value instead of its peak.
- Letting a strong control pass because the target run also passed.

</details>

<details>
<summary>Solution</summary>

Take `values[1:] - values[:-1]`, report the post-jump step, and gate the random control on its maximum value.

</details>


In [ ]:
def phase_transition_report(
    steps: t.Tensor,
    values: t.Tensor,
    *,
    metric_name: str = "mechanism_metric",
    min_jump: float = 0.25,
) -> PhaseTransitionReport:
    raise NotImplementedError()


def random_control_report(
    values: t.Tensor,
    *,
    metric_name: str = "random_control",
    max_allowed_value: float = 0.3,
) -> RandomControlReport:
    raise NotImplementedError()


tests.test_phase_transition_report_detects_largest_adjacent_jump(
    phase_transition_report
)
tests.test_random_control_report_rejects_overstrong_control(random_control_report)


## Exercise 4 - Developmental Comparisons

The toy trajectories include AR, JEPA, diffusion, Mamba-style curves, and a random control. The control should be visible in the report but excluded from earliest/latest model-family ordering.

> Difficulty: medium  
> Importance: medium

<details>
<summary>Expected output</summary>

```text
All tests in `test_developmental_comparison_excludes_random_control_from_ordering` passed!
```

</details>

<details>
<summary>Help - why not rank the random control?</summary>

The control is not a model family whose timing you want to compare. It is a falsification check for whether the family timing story is meaningful.

</details>

<details>
<summary>Common bugs</summary>

- Dropping the random control from the report entirely.
- Treating a missing stable step as zero.
- Including the random control in earliest/latest ordering.

</details>

<details>
<summary>Solution</summary>

Compute stable steps for every trajectory, filter the control out of ordering, then separately report whether the control stayed below threshold.

</details>


In [ ]:
def developmental_comparison_report(
    steps: t.Tensor,
    family_values: dict[str, t.Tensor],
    *,
    threshold: float = 0.6,
    min_consecutive: int = 2,
    control_name: str = "random_control",
) -> DevelopmentalComparisonReport:
    raise NotImplementedError()


tests.test_developmental_comparison_excludes_random_control_from_ordering(
    developmental_comparison_report
)


## Exercise 5 - Live Checkpoint Archaeology

Toy trajectories teach the report logic, but checkpoint archaeology is about saved training states. This exercise trains a tiny mod-13 addition model, saves checkpoints, reloads them before measuring, and compares the target run with a random-label control on the same schedule.

> Difficulty: medium-hard  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls` passed!
```

</details>

<details>
<summary>Help - why reload the checkpoint files?</summary>

Reloading proves the trajectory can be reconstructed from historical artifacts. Measuring only the current in-memory model would not be checkpoint archaeology.

</details>

<details>
<summary>Common bugs</summary>

- Saving only the final checkpoint.
- Measuring before reloading.
- Training the random-label control on a different checkpoint schedule.
- Describing complete finite-domain evaluation as OOD generalization.

</details>

<details>
<summary>Solution</summary>

Save every declared step, reload each file with `utils.checkpoint_metrics_from_file`, analyze the reloaded target trajectory, and require the random-label control to stay near chance.

</details>


In [ ]:
def _train_save_reload_modular_run(
    checkpoint_dir: Path,
    *,
    device: t.device,
    seed: int,
    random_labels: bool = False,
) -> dict:
    raise NotImplementedError()


def live_checkpoint_archaeology_smoke_test(
    checkpoint_root: Path | None = None,
    *,
    device: str | t.device = "cpu",
    seed: int = 0,
) -> dict:
    raise NotImplementedError()


tests.test_live_checkpoint_archaeology_smoke_test_trains_saves_reloads_and_controls(
    live_checkpoint_archaeology_smoke_test
)


## Exercise 6 - Notebook Contract and CUDA Report

The notebook contract is the CPU-feasible path students run while working. The committed CUDA report is produced by the section verification runner and records the same finite-table checkpoint archaeology on GPU.

> Difficulty: medium  
> Importance: high

<details>
<summary>Expected output</summary>

```text
All tests in `test_checkpoint_emergence_smoke_test` passed!
All tests in `test_phase_transition_smoke_test` passed!
All tests in `test_random_control_smoke_test` passed!
All tests in `test_developmental_comparison_smoke_test` passed!
All tests in `test_notebook_contract` passed!
All tests in `test_committed_gpu_report_records_real_checkpoint_preflight` passed!
```

</details>

<details>
<summary>Help - why read the committed CUDA report?</summary>

The notebook should stay fast and inspectable. The full acceptance path is regenerated by `scripts/run_extension_verification_reports.py --section 17.1`, then the notebook reads the committed JSON report and checks the same scoped evidence.

</details>

<details>
<summary>Common bugs</summary>

- Returning dataclass objects instead of JSON-serializable dicts.
- Dropping the live checkpoint path from `run_smoke_test`.
- Treating the CPU smoke path as a substitute for the CUDA report.

</details>

<details>
<summary>Solution</summary>

Expose small smoke-test dictionaries, read `verification_report.json` for the committed GPU metrics, and keep finite-domain scope explicit.

</details>


In [ ]:
def checkpoint_emergence_smoke_test() -> dict:
    raise NotImplementedError()


def phase_transition_smoke_test() -> dict:
    raise NotImplementedError()


def random_control_smoke_test() -> dict:
    raise NotImplementedError()


def developmental_comparison_smoke_test() -> dict:
    raise NotImplementedError()


def run_smoke_test(cpu: bool = True) -> dict:
    raise NotImplementedError()


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    raise NotImplementedError()


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    raise NotImplementedError()


tests.test_checkpoint_emergence_smoke_test(checkpoint_emergence_smoke_test)
tests.test_phase_transition_smoke_test(phase_transition_smoke_test)
tests.test_random_control_smoke_test(random_control_smoke_test)
tests.test_developmental_comparison_smoke_test(developmental_comparison_smoke_test)
tests.test_notebook_contract(run_smoke_test)
tests.test_committed_gpu_report_records_real_checkpoint_preflight()


## Signature Result

A convincing 17.1 result is not a single training curve. It is a bounded checkpoint report where the target run, saved files, reload metrics, stable threshold, phase jump, and random-label control all agree.

| Check | Passing result |
|---|---:|
| Complete finite table evaluated | `169 / 169` mod-13 pairs |
| Real checkpoints written and reloaded | `26` |
| First threshold crossing | step `30` |
| Stable threshold crossing | step `30` |
| Final target accuracy | `1.0` |
| Largest adjacent phase jump | `0.2781` |
| Random-label peak true-table accuracy | `0.1006` |
| Peak VRAM | `0.063 GB` |

<details>
<summary>Interpreting the result</summary>

This supports the scoped claim: for a generated finite mod-13 model organism, saved and reloaded checkpoints show stable table-accuracy emergence at step 30, a real adjacent jump, and a random-label control that does not look emergent.

</details>

## Limitations

Supported: deterministic toy trajectories, stable-threshold and phase-transition reports, random-label controls, CPU-feasible live checkpoint archaeology, and CUDA finite-table mod-13 save/reload evidence.

Not supported: large-model checkpoint archaeology, real transformer mechanism discovery, OOD generalization, causal localization inside the MLP, or real AR/JEPA/diffusion/Mamba training comparisons.

Deferred: real transformer checkpoint archaeology, activation-level mechanism localization, grokking curve replication, and checkpoint-difference circuit tracing.
